# WOSAC Baseline: Colab Smoke Test (01)

Goal: run one reproducible end-to-end smoke pipeline for the baseline pack.

Success criteria:
- runtime bootstrap succeeds without manual edits,
- config and run contract validate,
- baseline workflow entrypoint executes,
- metrics artifact is written to repo and Drive.


In [ ]:
# Step 1: Repo sync + deterministic runtime bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config

runtime_cfg = wosac_colab_runtime_config(
    repo_url=REPO_URL,
    repo_branch="main",
    repo_dir=REPO_DIR,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)
print("drive_status:", bootstrap.drive_status.detail)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load experiment config and compute config hash
import hashlib
import json

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "wosac-baseline"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")

cfg_str = json.dumps(cfg, sort_keys=True, indent=2)
cfg_hash = hashlib.sha256(cfg_str.encode("utf-8")).hexdigest()

print("experiment_slug:", EXPERIMENT_SLUG)
print("cfg_hash:", cfg_hash)
print(cfg_str)


In [ ]:
# Step 3: Run context and fast-fail validation
from datetime import datetime, timezone
from pathlib import Path

RUN = dict(cfg.get("run", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "wosac_baseline"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
N_SHARDS = int(RUN.get("n_shards", 1))
SHARD_ID = int(RUN.get("shard_id", 0))
RESUME_FROM_EXISTING = bool(RUN.get("resume_from_existing", True))
RUN_ENABLED = bool(RUN.get("run_enabled", True))

assert PERSIST_ROOT.strip(), "persist_root is required"
assert N_SHARDS >= 1, "n_shards must be >= 1"
assert 0 <= SHARD_ID < N_SHARDS, "shard_id must be in [0, n_shards)"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("RUN_PREFIX:", RUN_PREFIX)
print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("RESUME_FROM_EXISTING:", RESUME_FROM_EXISTING)
print("RUN_ENABLED:", RUN_ENABLED)


In [ ]:
# Step 4: Optional data-shard probe (set env var WOSAC_SAMPLE_TFRECORD to enable)
from pathlib import Path

sample_tfrecord = os.environ.get("WOSAC_SAMPLE_TFRECORD", "").strip()
data_probe = {
    "env_var": "WOSAC_SAMPLE_TFRECORD",
    "path": sample_tfrecord,
    "exists": False,
    "quick_probe_pass": False,
    "detail": "Set WOSAC_SAMPLE_TFRECORD to a TFRecord path for real data probe.",
}

if sample_tfrecord:
    p = Path(sample_tfrecord)
    data_probe["exists"] = p.exists()
    data_probe["quick_probe_pass"] = p.exists()
    data_probe["detail"] = "path_exists" if p.exists() else "path_missing"
else:
    # Keep smoke test runnable even without local dataset mount.
    data_probe["quick_probe_pass"] = True

print(data_probe)
quick_probe_pass = bool(data_probe["quick_probe_pass"])
assert quick_probe_pass, "Quick probe failed. Set a valid WOSAC_SAMPLE_TFRECORD path."


In [ ]:
# Step 5: Run baseline workflow entrypoint (thin orchestration call)
from src.workflows import run_wosac_baseline_flow

flow_bundle = run_wosac_baseline_flow(
    run_tag=RUN_TAG,
    shard_id=SHARD_ID,
    n_shards=N_SHARDS,
    n_rollouts_per_scenario=int(RUN.get("n_rollouts_per_scenario", 32)),
    num_history_seconds=int(RUN.get("num_history_seconds", 1)),
    num_future_seconds=int(RUN.get("num_future_seconds", 8)),
)

print("flow_summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))

metrics = {
    "realism_meta_metric": None,
    "simulated_collision_rate": None,
    "simulated_offroad_rate": None,
    "simulated_traffic_light_violation_rate": None,
}
metrics_note = "flow_scaffold_only" if flow_bundle.summary.get("status") == "todo" else "flow_executed"


In [ ]:
# Step 6: Write baseline metric artifact to repo and Drive
import subprocess

from pathlib import Path

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = "unknown"

payload = {
    "run_id": "baseline_v0",
    "status": "smoke_pass",
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "metrics": metrics,
    "flow_summary": flow_bundle.summary,
    "data_probe": data_probe,
    "notes": [
        "Smoke test validates pipeline contract and artifact writing.",
        "Integrate official evaluator outputs to replace null metric values.",
        f"flow_status={flow_bundle.summary.get('status')}",
        f"metrics_note={metrics_note}",
    ],
    "provenance": {
        "repo_commit": git_commit,
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

repo_metrics_path = Path("experiments/wosac-baseline/results/baseline_v0_metrics.json")
repo_metrics_path.parent.mkdir(parents=True, exist_ok=True)
repo_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

drive_metrics_path = persist_run_dir / "outputs" / "baseline_v0_metrics.json"
drive_metrics_path.parent.mkdir(parents=True, exist_ok=True)
drive_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("repo_metrics_path:", repo_metrics_path.resolve())
print("drive_metrics_path:", drive_metrics_path)
payload


## Next actions after smoke pass

1. Replace placeholder `run_wosac_baseline_flow` internals with actual one-shard inference and evaluator calls.
2. Populate non-null metric values from evaluator output.
3. Repeat in a second Colab session to confirm reproducibility before ablations.
